In [79]:
import networkx as nx
from bp.world import random_grid_world, Scenario

In [80]:
world = random_grid_world(
    rows=4,
    cols=4,
    demands={
        ((0, 0), (3, 3)): 10,
        ((1, 0), (1, 3)): 5,
    },
    seed=0,
)
G = world.network.graph

print(f"{world.total_population=}")
print(f"{world.total_demand((0, 0), (3, 3))=}")
print(f"{world.total_demand((1, 0), (1, 3))=}")

world.total_population=15
world.total_demand((0, 0), (3, 3))=10
world.total_demand((1, 0), (1, 3))=5


In [81]:
from itertools import pairwise
def edge_path(path):
    """Given a path consisting of vertices, return the corresponding edge-path"""
    return list(pairwise(path)) # list for state safety, can remove if careful though

In [82]:
import numpy as np

def debug_scenario(realized, omega, volumes):
    active_arcs = {a: t for a, t in realized.travel_time.items() if volumes[a] > 0}

    # (volumes[a] - 1) b/c travel_time holds average travel time including the next marginal traveler
    travel_time = sum(realized.travel_time[a] * (volumes[a] - 1) for a in active_arcs)
    average_travel_time = travel_time / world.total_population
    print(f"{travel_time=:.2f}")
    print(f"{average_travel_time=:.2f}")

    # slightly inaccurate b/c travel_time is off by one (see above)

    nominal_arc_time = sum(omega.travel_time[a] for a in active_arcs)
    realized_arc_time = sum(realized.travel_time[a] for a in active_arcs)
    average_arc_time = sum(active_arcs.values()) / len(active_arcs)
    arc_time_increase = realized_arc_time - nominal_arc_time
    average_arc_time_increase = arc_time_increase / len(active_arcs)
    print(f"{average_arc_time=:.2f} (along arcs w/ non-zero flow/volume)")
    print(f"{average_arc_time_increase=:.2f} (+{arc_time_increase / nominal_arc_time * 100:.2f}%)")

    flow = np.array([volumes.get(arc, 0) for arc in world.ordered_arcs], dtype=float)
    capacity_used = flow / world.network._arc_array("capacity") * 100
    average_capacity_used = np.average(capacity_used)
    most_capacity_used = np.max(capacity_used)
    print(f"{most_capacity_used=:.2f}%")
    print(f"{average_capacity_used=:.2f}%")

    # should sort then bin-search for `congested` b/c sorting but copy paste lazy
    congested = sum(realized.travel_time[a] > omega.travel_time[a] + 1e-9 for a in world.A)
    print(f"arcs whose travel time rose under congestion: {congested}/{len(world.A)}")

    most_congested = sorted(((a, omega.travel_time[a], realized.travel_time[a]) for a in world.A ), reverse=True, key=lambda x: x[2] - x[1])
    for i, ((orig, dest), t0, t1) in enumerate(most_congested[:congested], 1):
        print(f"{i}: {orig}->{dest}: +{(t1 - t0) / t0 * 100:.2f}%")

In [83]:
routes = {
    plr: edge_path(nx.shortest_path(G, plr.demand.origin, plr.demand.destination, weight="travel_time"))
    for plr in world.individuals
}

volumes = {arc: 0 for arc in world.A}
for route in routes.values():
    for arc in route:
        volumes[arc] += 1

realized_concurrent = world.realize(routes)
omega_concurrent = Scenario.from_world("nominal", world)

print("CONCURRENT:")
debug_scenario(realized_concurrent, omega_concurrent, volumes)

CONCURRENT:
travel_time=26429.79
average_travel_time=1761.99
average_arc_time=328.18 (along arcs w/ non-zero flow/volume)
average_arc_time_increase=322.50 (+5681.65%)
most_capacity_used=780.45%
average_capacity_used=67.75%
arcs whose travel time rose under congestion: 9/48
1: (2, 3)->(3, 3): +55652.05%
2: (0, 3)->(1, 3): +22824.33%
3: (0, 0)->(0, 1): +8026.95%
4: (1, 3)->(2, 3): +3980.65%
5: (0, 1)->(0, 2): +2861.53%
6: (0, 2)->(0, 3): +347.55%
7: (1, 2)->(1, 3): +108.19%
8: (1, 1)->(1, 2): +16.64%
9: (1, 0)->(1, 1): +18.13%


In [84]:
def dynamic_weight(orig, dest, _):
    return current_travel_times[(orig, dest)]

volumes = {arc: 0 for arc in world.A}
routes = {}
for plr in world.individuals:
    current_travel_times = world.network.actual_travel_times(volumes)
    route = edge_path(nx.shortest_path(G, plr.demand.origin, plr.demand.destination, weight=dynamic_weight))
    routes[plr] = route
    for arc in route:
        volumes[arc] += 1

realized_sequential = world.realize(routes)
omega_sequential = Scenario.from_world("nominal", world)

print("SEQUENTIAL:")
debug_scenario(realized_sequential, omega_sequential, volumes)

SEQUENTIAL:
travel_time=573.83
average_travel_time=38.26
average_arc_time=9.57 (along arcs w/ non-zero flow/volume)
average_arc_time_increase=3.49 (+57.41%)
most_capacity_used=312.18%
average_capacity_used=50.60%
arcs whose travel time rose under congestion: 21/48
1: (2, 3)->(3, 3): +1424.69%
2: (0, 0)->(0, 1): +205.49%
3: (1, 2)->(1, 3): +108.19%
4: (0, 0)->(1, 0): +82.50%
5: (2, 2)->(3, 2): +69.42%
6: (3, 2)->(3, 3): +51.51%
7: (1, 0)->(1, 1): +118.83%
8: (0, 1)->(0, 2): +73.26%
9: (2, 1)->(2, 2): +28.75%
10: (1, 1)->(1, 2): +16.64%
11: (0, 3)->(1, 3): +36.52%
12: (1, 3)->(2, 3): +6.37%
13: (0, 2)->(1, 2): +7.85%
14: (2, 0)->(2, 1): +33.17%
15: (2, 2)->(2, 3): +1.88%
16: (1, 2)->(2, 2): +4.38%
17: (1, 0)->(2, 0): +2.25%
18: (2, 1)->(3, 1): +0.96%
19: (1, 1)->(2, 1): +2.12%
20: (0, 2)->(0, 3): +0.56%
21: (3, 1)->(3, 2): +0.05%
